In [ ]:
import pandas as pd
import time
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

df = pd.read_csv("Water Quality Prediction.csv")

df_0 = df[df["Target"] == 0].sample(n=20000, random_state=42)
df_1 = df[df["Target"] == 1].sample(n=5000, random_state=42)
df_balanced = pd.concat([df_0, df_1]).sample(frac=1, random_state=42).reset_index(drop=True)

if "Index" in df_balanced.columns:
    df_balanced.drop(columns="Index", inplace=True)

target = "Target"
X = df_balanced.drop(columns=target)
y = df_balanced[target]

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(include=["float64", "int64"]).columns.tolist()

imputation_strategies = {
    "mean": SimpleImputer(strategy="mean"),
    "median": SimpleImputer(strategy="median"),
    "knn": KNNImputer(n_neighbors=5)
}

models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(verbose=-1, random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42, train_dir="catboost_tmp"),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []

for strategy_name, imputer in imputation_strategies.items():
    print(f"\n--- Импутация: {strategy_name} ---")
    numeric_transformer = Pipeline([
        ("imputer", imputer),
        ("scaler", StandardScaler())
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ])

    for model_name, model in models.items():
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", model)
        ])

        start_time = time.time()
        scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy", n_jobs=-1, error_score='raise')
        elapsed = time.time() - start_time
        avg_score = scores.mean()

        print(f"{model_name:<20} | Accuracy: {avg_score:.4f} | Time: {elapsed:.2f} sec")
        results.append((strategy_name, model_name, avg_score, elapsed))


--- Импутация: mean ---
RandomForest         | Accuracy: 0.8566 | Time: 9.68 sec
LogisticRegression   | Accuracy: 0.8192 | Time: 0.22 sec
XGBoost              | Accuracy: 0.8392 | Time: 0.89 sec
LightGBM             | Accuracy: 0.8529 | Time: 0.79 sec
CatBoost             | Accuracy: 0.8469 | Time: 11.74 sec
GradientBoosting     | Accuracy: 0.8359 | Time: 12.51 sec

--- Импутация: median ---
RandomForest         | Accuracy: 0.8587 | Time: 10.17 sec
LogisticRegression   | Accuracy: 0.8193 | Time: 0.26 sec
XGBoost              | Accuracy: 0.8409 | Time: 0.82 sec
LightGBM             | Accuracy: 0.8528 | Time: 0.81 sec
CatBoost             | Accuracy: 0.8467 | Time: 14.02 sec
GradientBoosting     | Accuracy: 0.8352 | Time: 13.52 sec

--- Импутация: knn ---
RandomForest         | Accuracy: 0.8538 | Time: 23.70 sec
LogisticRegression   | Accuracy: 0.8195 | Time: 13.48 sec
XGBoost              | Accuracy: 0.8403 | Time: 13.89 sec
LightGBM             | Accuracy: 0.8495 | Time: 14.59 sec
Cat